# Using an API to Communicate with a Model

**AI for All Workshop**

You are a **client** sending messages to a **service**. The model runs on
ASU Research Computing's servers; you send it text over HTTPS and read the
text it returns. You do not install a model or run any training.

**The API:** the ASU Research Computing LLM gateway
(`https://openai.rc.asu.edu/v1`). It follows the OpenAI API format, so the
standard `openai` Python package works with it as-is.

**The plan** (your 15 minutes of experiment time is in section 4):

1. Set your key (kept out of your code)
2. First call — get a real response back  ← *checkpoint*
3. Build the research tool — 12 abstracts to a structured CSV
4. Experiments — vary prompts and models, then try your own data

The 12 abstracts are already in this notebook (section 3a). There is nothing
to download.


## 1 - Set your key

This lesson stores your credentials in a `.env` file - a plain-text file with
one `NAME=value` per line. If you don't have one yet, create a file named
`.env` (no extension) in this folder, next to the notebook, with these two
lines (replace `<your-key>` with your key):

```
OPENAI_API_KEY=<your-key>
OPENAI_BASE_URL=https://openai.rc.asu.edu/v1
```

Then run this cell. It loads the file with `load_dotenv()` and confirms the
key is present, so the key never sits in your code.

**Treat your key like a password.** Don't share it, don't commit it to Git
(keep `.env` out of version control - see the lesson's Setup page).

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from the .env file
load_dotenv()

# Retrieve the API key and base URL
key = os.getenv("OPENAI_API_KEY")
base = os.getenv("OPENAI_BASE_URL")

if not key or not base:
    missing = ", ".join(n for n, v in
                        (("OPENAI_API_KEY", key), ("OPENAI_BASE_URL", base)) if not v)
    raise SystemExit(
        f"Missing {missing}. Create a .env file in this folder with these two "
        "lines, then re-run this cell:\n"
        "    OPENAI_API_KEY=<your-key>\n"
        "    OPENAI_BASE_URL=https://openai.rc.asu.edu/v1"
    )

# Mask the key so we can show it is present without revealing it
print(f"key set:   {key[:4]}...{key[-4:]}  ({len(key)} chars)")
print(f"base url:  {base}")

## 2 - First call

The whole API in a few lines. `messages` is a list of `{role, content}` pairs,
and the answer comes back at `choices[0].message.content`.

The next cell picks one of the three models this lesson prefers, at random,
from your key's live list, so you do not need to know the exact name. It also
prints up to eight models your key can use - you will try a couple of them in
the experiments.

In [ ]:
from openai import OpenAI
import random

client = OpenAI()   # reads OPENAI_API_KEY and OPENAI_BASE_URL from the environment

# --- Pick a model at random ---------------------------------------------------
PREFERRED = ["qwen36-27b", "muse-glimmer-30b", "gemma4-31b-it"]
try:
    available = [m.id for m in client.models.list()]
except Exception as e:
    available = []
    print(f"Could not list models: {e}")

# A preferred name can be a prefix of the full model ID, so match by
# substring. Then pick one of the matches at random.
matches = [a for a in available if any(p in a for p in PREFERRED)]
if matches:
    MODEL = random.choice(matches)
elif available:
    MODEL = available[0]   # none of the three is available; use the first
else:
    raise SystemExit(
        "No model could be selected. Set one manually, e.g.\n"
        "    MODEL = 'llama3.1'\n"
        "then re-run."
    )
print(f"Using model: {MODEL}  (picked at random)")
if available:
    print(f"Your key can use {len(available)} model(s). Try others in the experiments:")
    for a in available[:8]:
        print(f"   - {a}")

# --- The first request --------------------------------------------------------
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain what an API is, in one sentence."}],
)
print("\n--- MODEL SAYS ---")
print(resp.choices[0].message.content)
print(f"\n(tokens used: {resp.usage.total_tokens if resp.usage else '?'})")

## 3 - Build the research tool

The task: run one question over a list of inputs and collect the answers into
a table. Here that is 12 arXiv abstracts (already in section 3a) in, one row
per abstract out — a summary, the method, and the result.

The *system prompt* tells the model to answer in **strict JSON** with the keys
`summary`, `method`, and `result`. That is **structured output**: you specify
the shape of the answer, so the result is columns you can load into a
spreadsheet rather than a paragraph you have to read.

Run the cells in this order: load the data (3a), run the batch (3b), save and
inspect (3c).


### 3a - Load the 12 abstracts (already in the notebook)


In [ ]:
import json

ABSTRACTS = [
  {
    "field": "Machine Learning (CS)",
    "title": "Learning Active Subspaces and Discovering Important Features with Gaussian Radial Basis Functions Neural Networks",
    "abstract": "Providing a model that achieves a strong predictive performance and is simultaneously interpretable by humans is one of the most difficult challenges in machine learning research due to the conflicting nature of these two objectives. To address this challenge, we propose a modification of the radial basis function neural network model by equipping its Gaussian kernel with a learnable precision matrix. We show that precious information is contained in the spectrum of the precision matrix that can be extracted once the training of the model is completed. In particular, the eigenvectors explain the directions of maximum sensitivity of the model revealing the active subspace and suggesting potential applications for supervised dimensionality reduction. At the same time, the eigenvectors highlight the relationship in terms of absolute variation between the input and the latent variables, thereby allowing us to extract a ranking of the input variables based on their importance to the prediction task enhancing the model interpretability. We conducted numerical experiments for regression, classification, and feature selection tasks, comparing our model against popular machine learning models, the state-of-the-art deep learning-based embedding feature selection techniques, and a transformer model for tabular data. Our results demonstrate that the proposed model does not only yield an attractive prediction performance compared to the competitors but also provides meaningful and interpretable results that potentially could assist the decision-making process in real-world applications. A PyTorch implementation of the model is available on GitHub at the following link. https://github.com/dannyzx/Gaussian-RBFNN",
    "id": "2307.05639v2"
  },
  {
    "field": "Machine Learning (CS)",
    "title": "Hierarchical Attentional Hybrid Neural Networks for Document Classification",
    "abstract": "Document classification is a challenging task with important applications. The deep learning approaches to the problem have gained much attention recently. Despite the progress, the proposed models do not incorporate the knowledge of the document structure in the architecture efficiently and not take into account the contexting importance of words and sentences. In this paper, we propose a new approach based on a combination of convolutional neural networks, gated recurrent units, and attention mechanisms for document classification tasks. The main contribution of this work is the use of convolution layers to extract more meaningful, generalizable and abstract features by the hierarchical representation. The proposed method in this paper improves the results of the current attention-based approaches for document classification.",
    "id": "1901.06610v2"
  },
  {
    "field": "Biology",
    "title": "Learning differential module networks across multiple experimental conditions",
    "abstract": "Module network inference is a statistical method to reconstruct gene regulatory networks, which uses probabilistic graphical models to learn modules of coregulated genes and their upstream regulatory programs from genome-wide gene expression and other omics data. Here we review the basic theory of module network inference, present protocols for common gene regulatory network reconstruction scenarios based on the Lemon-Tree software, and show, using human gene expression data, how the software can also be applied to learn differential module networks across multiple experimental conditions.",
    "id": "1711.08927v2"
  },
  {
    "field": "Biology",
    "title": "Gene regulatory network inference: an introductory survey",
    "abstract": "Gene regulatory networks are powerful abstractions of biological systems. Since the advent of high-throughput measurement technologies in biology in the late 90s, reconstructing the structure of such networks has been a central computational problem in systems biology. While the problem is certainly not solved in its entirety, considerable progress has been made in the last two decades, with mature tools now available. This chapter aims to provide an introduction to the basic concepts underpinning network inference tools, attempting a categorisation which highlights commonalities and relative strengths. While the chapter is meant to be self-contained, the material presented should provide a useful background to the later, more specialised chapters of this book.",
    "id": "1801.04087v2"
  },
  {
    "field": "Materials Science",
    "title": "The Efficiency Limit of CH3NH3PbI3 Perovskite Solar Cells",
    "abstract": "With the consideration of photon recycling effect, the efficiency limit of methylammonium lead iodide (CH3NH3PbI3) perovskite solar cells is predicted by a detailed balance model. To obtain convincing predictions, both AM 1.5 spectrum of Sun and experimentally measured complex refractive index of perovskite material are employed in the detailed balance model. The roles of light trapping and angular restriction in improving the maximal output power of thin-film perovskite solar cells are also clarified. The efficiency limit of perovskite cells (without the angular restriction) is about 31%, which approaches to Shockley-Queisser limit (33%) achievable by gallium arsenide (GaAs) cells. Moreover, the Shockley-Queisser limit could be reached with a 200 nm-thick perovskite solar cell, through integrating a wavelength-dependent angular-restriction design with a textured light-trapping structure. Additionally, the influence of the trap-assisted nonradiative recombination on the device efficiency is investigated. The work is fundamentally important to high-performance perovskite photovoltaics.",
    "id": "1506.09003v1"
  },
  {
    "field": "Materials Science",
    "title": "Towards the maximum efficiency design of a perovskite solar cell by material properties tuning: A multidimensional approach",
    "abstract": "To obtain significant increases in the Power Conversion Efficiency (PCE) of solar cells, future cell research and development should be based on the concomitant improvement of multiple material properties, rather than on the state-of-the-art one or two-dimensional improvements. In this context, researchers should know, which combined material properties and cell design parameters lead to the highest efficiency increase. For the same objective, it should also be known which relationships in-between these variables have to be adjusted. Such knowledge becomes available by simulation and numerical optimization, which we present for a Perovskite Solar Cell(PSC)in a hypercube space of variables.",
    "id": "1711.03818v2"
  },
  {
    "field": "Environmental Science",
    "title": "Predicting concentration levels of air pollutants by transfer learning and recurrent neural network",
    "abstract": "Air pollution (AP) poses a great threat to human health, and people are paying more attention than ever to its prediction. Accurate prediction of AP helps people to plan for their outdoor activities and aids protecting human health. In this paper, long-short term memory (LSTM) recurrent neural networks (RNNs) have been used to predict the future concentration of air pollutants (APS) in Macau. Additionally, meteorological data and data on the concentration of APS have been utilized. Moreover, in Macau, some air quality monitoring stations (AQMSs) have less observed data in quantity, and, at the same time, some AQMSs recorded less observed data of certain types of APS. Therefore, the transfer learning and pre-trained neural networks have been employed to assist AQMSs with less observed data to build a neural network with high prediction accuracy. The experimental sample covers a period longer than 12-year and includes daily measurements from several APS as well as other more classical meteorological values. Records from five stations, four out of them are AQMSs and the remaining one is an automatic weather station, have been prepared from the aforesaid period and eventually underwent to computational intelligence techniques to build and extract a prediction knowledge-based system. As shown by experimentation, LSTM RNNs initialized with transfer learning methods have higher prediction accuracy; it incurred shorter training time than randomly initialized recurrent neural networks.",
    "id": "2502.01654v1"
  },
  {
    "field": "Environmental Science",
    "title": "Urban Air Pollution Forecasting: a Machine Learning Approach leveraging Satellite Observations and Meteorological Forecasts",
    "abstract": "Air pollution poses a significant threat to public health and well-being, particularly in urban areas. This study introduces a series of machine-learning models that integrate data from the Sentinel-5P satellite, meteorological conditions, and topological characteristics to forecast future levels of five major pollutants. The investigation delineates the process of data collection, detailing the combination of diverse data sources utilized in the study. Through experiments conducted in the Milan metropolitan area, the models demonstrate their efficacy in predicting pollutant levels for the forthcoming day, achieving a percentage error of around 30%. The proposed models are advantageous as they are independent of monitoring stations, facilitating their use in areas without existing infrastructure. Additionally, we have released the collected dataset to the public, aiming to stimulate further research in this field. This research contributes to advancing our understanding of urban air quality dynamics and emphasizes the importance of amalgamating satellite, meteorological, and topographical data to develop robust pollution forecasting models.",
    "id": "2405.19901v1"
  },
  {
    "field": "Medicine / Health",
    "title": "Segmentation-Renormalized Deep Feature Modulation for Unpaired Image Harmonization",
    "abstract": "Deep networks are now ubiquitous in large-scale multi-center imaging studies. However, the direct aggregation of images across sites is contraindicated for downstream statistical and deep learning-based image analysis due to inconsistent contrast, resolution, and noise. To this end, in the absence of paired data, variations of Cycle-consistent Generative Adversarial Networks have been used to harmonize image sets between a source and target domain. Importantly, these methods are prone to instability, contrast inversion, intractable manipulation of pathology, and steganographic mappings which limit their reliable adoption in real-world medical imaging. In this work, based on an underlying assumption that morphological shape is consistent across imaging sites, we propose a segmentation-renormalized image translation framework to reduce inter-scanner heterogeneity while preserving anatomical layout. We replace the affine transformations used in the normalization layers within generative networks with trainable scale and shift parameters conditioned on jointly learned anatomical segmentation embeddings to modulate features at every level of translation. We evaluate our methodologies against recent baselines across several imaging modalities (T1w MRI, FLAIR MRI, and OCT) on datasets with and without lesions. Segmentation-renormalization for translation GANs yields superior image harmonization as quantified by Inception distances, demonstrates improved downstream utility via post-hoc segmentation accuracy, and improved robustness to translation perturbation and self-adversarial attacks.",
    "id": "2102.06315v2"
  },
  {
    "field": "Medicine / Health",
    "title": "PSIGAN: Joint probabilistic segmentation and image distribution matching for unpaired cross-modality adaptation based MRI segmentation",
    "abstract": "We developed a new joint probabilistic segmentation and image distribution matching generative adversarial network (PSIGAN) for unsupervised domain adaptation (UDA) and multi-organ segmentation from magnetic resonance (MRI) images. Our UDA approach models the co-dependency between images and their segmentation as a joint probability distribution using a new structure discriminator. The structure discriminator computes structure of interest focused adversarial loss by combining the generated pseudo MRI with probabilistic segmentations produced by a simultaneously trained segmentation sub-network. The segmentation sub-network is trained using the pseudo MRI produced by the generator sub-network. This leads to a cyclical optimization of both the generator and segmentation sub-networks that are jointly trained as part of an end-to-end network. Extensive experiments and comparisons against multiple state-of-the-art methods were done on four different MRI sequences totalling 257 scans for generating multi-organ and tumor segmentation. The experiments included, (a) 20 T1-weighted (T1w) in-phase mdixon and (b) 20 T2-weighted (T2w) abdominal MRI for segmenting liver, spleen, left and right kidneys, (c) 162 T2-weighted fat suppressed head and neck MRI (T2wFS) for parotid gland segmentation, and (d) 75 T2w MRI for lung tumor segmentation. Our method achieved an overall average DSC of 0.87 on T1w and 0.90 on T2w for the abdominal organs, 0.82 on T2wFS for the parotid glands, and 0.77 on T2w MRI for lung tumors.",
    "id": "2007.09465v2"
  },
  {
    "field": "Energy",
    "title": "Lasso estimation for GEFCom2014 probabilistic electric load forecasting",
    "abstract": "We present a methodology for probabilistic load forecasting that is based on lasso (least absolute shrinkage and selection operator) estimation. The model considered can be regarded as a bivariate time-varying threshold autoregressive(AR) process for the hourly electric load and temperature. The joint modeling approach incorporates the temperature effects directly, and reflects daily, weekly, and annual seasonal patterns and public holiday effects. We provide two empirical studies, one based on the probabilistic load forecasting track of the Global Energy Forecasting Competition 2014 (GEFCom2014-L), and the other based on another recent probabilistic load forecasting competition that follows a setup similar to that of GEFCom2014-L. In both empirical case studies, the proposed methodology outperforms two multiple linear regression based benchmarks from among the top eight entries to GEFCom2014-L.",
    "id": "1603.01376v1"
  },
  {
    "field": "Energy",
    "title": "A comparative assessment of deep learning models for day-ahead load forecasting: Investigating key accuracy drivers",
    "abstract": "Short-term load forecasting (STLF) is vital for the effective and economic operation of power grids and energy markets. However, the non-linearity and non-stationarity of electricity demand as well as its dependency on various external factors renders STLF a challenging task. To that end, several deep learning models have been proposed in the literature for STLF, reporting promising results. In order to evaluate the accuracy of said models in day-ahead forecasting settings, in this paper we focus on the national net aggregated STLF of Portugal and conduct a comparative study considering a set of indicative, well-established deep autoregressive models, namely multi-layer perceptrons (MLP), long short-term memory networks (LSTM), neural basis expansion coefficient analysis (N-BEATS), temporal convolutional networks (TCN), and temporal fusion transformers (TFT). Moreover, we identify factors that significantly affect the demand and investigate their impact on the accuracy of each model. Our results suggest that N-BEATS consistently outperforms the rest of the examined models. MLP follows, providing further evidence towards the use of feed-forward networks over relatively more sophisticated architectures. Finally, certain calendar and weather features like the hour of the day and the temperature are identified as key accuracy drivers, providing insights regarding the forecasting approach that should be used per case.",
    "id": "2302.12168v2"
  }
]

print(f"Loaded {len(ABSTRACTS)} abstracts:")
for i, a in enumerate(ABSTRACTS, 1):
    print(f"  {i:2d}. [{a['field']}] {a['title'][:70]}")

### 3b - Run the batch (one API call per abstract)

This is the loop you will reuse on your own data. It prints progress as it
goes. Twelve calls usually take 30 to 90 seconds, depending on the model. If a
response is not valid JSON, the cell tries once more. Some models wrap the JSON
object in code fences; the cell removes them before it parses.


In [ ]:
import json, time

SYSTEM_PROMPT = (
    "You are a research literature assistant. Read the abstract and respond with "
    "ONLY a valid JSON object (no markdown, no commentary) with exactly these keys: "
    '{"summary": "<one-sentence summary>", '
    '"method": "<the main method or approach, one short phrase>", '
    '"result": "<the key result or contribution, one short phrase>"}. '
    'If a detail is not stated in the abstract, set it to "not stated". '
    "Return ONLY the JSON object."
)

def extract(record):
    # One model call for one abstract -> dict (or a PARSE-FAILED marker).
    user = (f"Field: {record['field']}\nTitle: {record['title']}\n"
            f"Abstract: {record['abstract']}")
    for attempt in range(2):   # one retry if the JSON is malformed
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user},
            ],
            temperature=0,
        )
        text = resp.choices[0].message.content.strip()
        # Strip any ```json ... ``` fence some models add around the object
        t = text
        if t.startswith("```"):
            t = t.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            return json.loads(t)
        except json.JSONDecodeError:
            if attempt == 0:
                continue
            return {"summary": "PARSE FAILED", "method": text[:80], "result": ""}

rows, failures = [], 0
for i, rec in enumerate(ABSTRACTS, 1):
    row = extract(rec)
    ok = bool(row) and row.get("summary") != "PARSE FAILED"
    if not ok:
        failures += 1
        rows.append({"field": rec["field"], "title": rec["title"], "arxiv_id": rec["id"],
                     "summary": "PARSE FAILED", "method": "", "result": ""})
        print(f"  {i:2d}/{len(ABSTRACTS)}  {rec['field']:<22} -> PARSE FAILED (kept raw snippet)")
    else:
        rows.append({"field": rec["field"], "title": rec["title"], "arxiv_id": rec["id"],
                     **{k: row.get(k, "not stated") for k in ("summary", "method", "result")}})
        print(f"  {i:2d}/{len(ABSTRACTS)}  {rec['field']:<22} -> ok")

print(f"\nDone. {len(rows)} rows, {failures} parse failure(s).")

### 3c - Save & inspect the CSV

In [ ]:
import csv

out = "triage_table.csv"
fields = ["field", "title", "arxiv_id", "summary", "method", "result"]
with open(out, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    w.writerows(rows)
print(f"Wrote {len(rows)} rows to {out}\n")

# Show a readable slice
for r in rows[:4]:
    print(f"[{r['field']}] {r['title'][:55]}")
    print(f"   summary: {r['summary'][:100]}")
    print(f"   method:  {r['method'][:80]}")
    print()

## 4 - Experiments

Run whichever cells fit your research. Each one changes exactly ONE variable - that's
how you learn what actually matters.

**Before Experiment 4:** do not send sensitive/regulated data (health records,
export-controlled work, SSNs, personal identifiers). If unsure, ask your institution
first.

### Experiment 1 - Vary the prompt (same abstract, 3 asks)

In [ ]:
rec = ABSTRACTS[0]   # pick any index 0..11

asks = [
    "Summarize this abstract in one sentence.",
    "Summarize this abstract in one sentence, then list up to 3 limitations the authors mention or that are implied.",
    "Explain the core idea of this abstract to a first-year PhD student in a different field. Keep it under 80 words.",
]
for i, ask in enumerate(asks, 1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful research assistant."},
            {"role": "user", "content": f"{ask}\n\nAbstract: {rec['abstract']}"},
        ],
    )
    print(f"--- Ask {i}: {ask[:60]}...")
    print(resp.choices[0].message.content.strip()[:300])
    print()

### Experiment 2 - Vary the model (same request, 2 models)

Set `B` to a second model ID from the list printed in Section 2. Leave `A` as the
current model. Watch style, length, and what each gets right or wrong.

In [ ]:
A = MODEL
B = None   # <-- set me to another model ID, e.g. B = 'gpt-4o' (must be in your list)

if not B:
    raise SystemExit("Set B = 'some-model-id' (from the list in Section 2) and re-run.")

prompt = ("In under 60 words, what is the main method and the key result of this "
          f"abstract?\n\nAbstract: {ABSTRACTS[1]['abstract']}")

for label, m in (("Model A", A), ("Model B", B)):
    try:
        resp = client.chat.completions.create(
            model=m,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        print(f"=== {label} ({m}) ===")
        print(resp.choices[0].message.content.strip())
        print()
    except Exception as e:
        print(f"=== {label} ({m}) === FAILED: {e}\n")

### Experiment 3 - Tighten the structure (add a field to the schema)

In [ ]:
SYSTEM2 = (
    "Respond with ONLY a valid JSON object (no other text) with exactly these keys: "
    '{"summary": "<one sentence>", '
    '"method": "<main method, short>", '
    '"result": "<key result, short>", '
    '"reproducible": "yes | no | maybe", '
    '"confidence": "high | medium | low"}. '
    "Use 'not stated' where the abstract gives no information. Return ONLY the JSON."
)

for rec in ABSTRACTS[:3]:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM2},
            {"role": "user", "content": f"Title: {rec['title']}\nAbstract: {rec['abstract']}"},
        ],
        temperature=0,
    )
    print(rec["title"][:60])
    print(resp.choices[0].message.content.strip())
    print()

### Experiment 4 - Your own data

Paste 3-5 short paragraphs from your own work into `MY_TEXTS` below and run the same
batch pattern. **Only send data that is fine to send** (see the warning above).

Stretch goal: add your own key to the JSON schema and watch the model fill it.

In [ ]:
MY_TEXTS = [
    "Paste paragraph 1 here",
    "Paste paragraph 2 here",
    "Paste paragraph 3 here",
]

if all(t.startswith("Paste") for t in MY_TEXTS):
    raise SystemExit("Replace the MY_TEXTS list with your own text, then re-run.")

for i, text in enumerate(MY_TEXTS, 1):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Text: {text}"},
        ],
        temperature=0,
    )
    print(f"--- Text {i} ---")
    print(resp.choices[0].message.content.strip())
    print()

## Where to go next

- **Streaming:** `client.chat.completions.create(..., stream=True)` sends the
  answer as it is generated, so you can watch it arrive.
- **Function and tool calling:** let the model call your own Python functions.
- **RAG (retrieval-augmented generation):** ground answers in your own
  documents. The *A Simple RAG Example* episode (after Experiments) builds
  one with the ChromaDB vector database; Purdue's AnvilGPT does the same
  out of the box.
- **Tools that use the same key and endpoint** (the format is OpenAI-compatible,
  so they work without changes):
  - **OpenCode** — a terminal and desktop assistant. ASU Research Computing has
    a setup guide, and Voyager generates the provider config for you.
  - **VS Code** — Chat, Manage Language Models, Add Models, Custom Endpoint:
    paste `https://openai.rc.asu.edu/v1` and your key.
  - **Jupyter AI** — the `%ai` magic inside notebooks.
- **Reference:** the ASU Research Computing API documentation is at
  <https://docs.rc.asu.edu/ai/api>. Purdue's AnvilGPT is at
  <https://anvilgpt.rcac.purdue.edu>.